# VAW Phase 1 — Colab run: M8a (person detection + pose + ByteTrack)
Needs a **GPU runtime** (Runtime -> Change runtime type -> T4 GPU) and notebook 01 already run (M2 output in Drive).
Cells only; logic lives in `src/assm/track_poses.py`. Resumable: re-running skips clips already tracked.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/KRITHIKSAI16/Violence-Against-Women.git"
if not os.path.isdir('/content/VAW'):
    !git clone -q {url} /content/VAW
else:
    !git -C /content/VAW pull -q {url} main
%cd /content/VAW
!pip install -q -r requirements.txt
!nvidia-smi -L
!python -m pytest -q

## Run M8a on the full cleaned corpus
One `.npz` per clip is written to `VAW_results/tracks/<Category>/`. A preview jpg per clip goes to `VAW_results/previews/` for a quick visual check.

In [ ]:
!python -m src.assm.track_poses --config configs/colab.yaml --preview-dir /content/drive/MyDrive/VAW_results/previews

## Sanity check: clips where nobody (or only one person) was tracked

In [ ]:
import glob, numpy as np
rows = []
for f in sorted(glob.glob('/content/drive/MyDrive/VAW_results/tracks/*/*.npz')):
    z = np.load(f)
    ids_per_frame = np.bincount(z['frame_idx'], minlength=int(z['n_frames']))
    rows.append((f.split('/')[-2], f.split('/')[-1][:-4], int(z['n_frames']), int((ids_per_frame >= 2).sum())))
print(len(rows), 'clips tracked')
for cat, cid, n, two in rows:
    if two == 0:
        print('NO 2+ PEOPLE:', cat, cid, f'({n} frames)')